# Inspecting FIPI before data selection

In this research, I use FIPI, or Fine-Grained Indirect Prompt Injection Dataset, as the source of training data for all 8 selection methods. Before selecting those data, I first examine what each prompt pair contains, how the injection is inserted, and whether the training & testing splits share the same original tasks.

By a 'prompt pair', I refer to two prompts: one is the benign/safe prompt, another contains an inserted injection instruction under the same user task.

- [The training and testing data](#The-training-and-testing-data)
- [How many prompts and injection methods are there?](#How-many-prompts-and-injection-methods-are-there?)
- [A training pair example](#A-training-pair-example)
- [Where is the injection inserted?](#Where-is-the-injection-inserted?)
- [What needs to be checked before using these data?](#What-needs-to-be-checked-before-using-these-data?)


## The training and testing data

I start with FIPI's original training and testing files, with the table variables `training` and `testing` containing their respective prompt pairs. Each record already provides the user instruction, original content and injected content, so these are the two versions that I will compare throughout this inspection.


In [ ]:
# inspect what each FIPI pair contains before selecting training data

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

# find the project root so the same data can be opened from the notebook or repo
root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'data/FIPI/raw').is_dir())
data_folder = root / 'data/FIPI/raw'

# keep the original training and testing splits separate for this inspection
training = pd.DataFrame(json.loads((data_folder / 'train_100000.json').read_text()))
testing = pd.DataFrame(json.loads((data_folder / 'test_10000.json').read_text()))

# show the full prompt text to inspect exactly where the injection appears
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

print(f'Data folder: {data_folder}')


## How many prompts and injection methods are there?

I count both the total prompt pairs and the distinct source prompts, as several pairs can use the same original task with different injection instructions. Meaning the number of pairs does not directly tell us how many different source tasks the dataset contains.

The first table compares those counts across training and testing, while the second shows how many pairs use each injection method.


In [ ]:
# count pairs and source prompts separately, since different injections can share one original task
counts = []
for split, data in [('training', training), ('testing', testing)]:
    # count instructions containing only spaces as empty instructions
    counts.append({'split': split, 'injection records': len(data), 'source prompts': data['user_prompt_id'].nunique(), 'empty task instructions': data['user_instruction'].str.strip().eq('').sum()})

display(pd.DataFrame(counts).set_index('split'))

# compare how often each injection method appears in training and testing
# use zero where a method is absent from one split
methods = pd.DataFrame({'training': training['attack_method'].value_counts(), 'testing': testing['attack_method'].value_counts()}).fillna(0).astype(int)
display(methods.rename_axis('injection method'))


## A training pair example

For this example, the variable `row_index` chooses one training record, and I put its safe and injection prompts beside each other to see what changed.

The source field `witness` contains the answer requested by the injection (such as 'blue' for a question about the ocean's color), which I later use to check whether the model produced the injected answer. The user instruction remains the same in both versions.


In [ ]:
# row_index chooses which training pair to inspect
row_index = 0
example = training.iloc[row_index]

print('Task:', example['user_instruction'])
print('Injection method:', example['attack_method'])
print('Inserted request:', example['attack_instruction'])
# witness is the answer requested by the injection, such as blue for the ocean question
print('Expected answer to the inserted request:', example['witness'])

# compare both prompt versions under the same user instruction
pair = pd.DataFrame({'condition': ['safe', 'injection'], 'task instruction': [example['user_instruction']] * 2, 'supplied content': [example['clean_data'], example['injected_data']]})
display(pair.set_index('condition'))


## Where is the injection inserted?

FIPI also records the character positions of each injection, so I use the source fields `start` and `end` to extract that exact part of the prompt and compare it with the original injection instruction.

The ending position includes its final character, meaning the Python slice needs `end + 1` to include it. Apart from checking the example above, I repeat this comparison across all training and testing records.


In [ ]:
# read the stored character positions for the selected injection
start = example['injected_position']['start']
end = example['injected_position']['end']
# add one because FIPI includes the end character while Python slices stop just before it
insertion = example['injected_data'][start:end + 1]

# repr makes line breaks visible when inspecting the inserted instruction
print('Inserted text:', repr(insertion))
print('Matches the stored instruction:', insertion == example['attack_instruction'])

# repeat the position check across every training and testing pair
span_counts = []
for split, data in [('training', training), ('testing', testing)]:
    matches = 0
    for row in data.itertuples():
        start = row.injected_position['start']
        end = row.injected_position['end']
        # count a match only when the positions are in bounds and the extracted text matches exactly
        matches += 0 <= start <= end < len(row.injected_data) and row.injected_data[start:end + 1] == row.attack_instruction
    span_counts.append({'split': split, 'matching spans': matches, 'records': len(data)})

display(pd.DataFrame(span_counts).set_index('split'))


## What needs to be checked before using these data?

Before using FIPI for data selection, I check whether the training and testing splits share source prompt IDs, task/content pairs or the same original text, as well as whether any user instructions are empty.

I also check if the answer requested by the injection already appears in the user instruction, since the model could then produce that answer while following the original task. The final table gives examples of records with missing task instructions.


In [ ]:
# count source prompt IDs and original texts shared between training and testing
shared_ids = set(training['user_prompt_id']) & set(testing['user_prompt_id'])
shared_content = set(training['clean_data']) & set(testing['clean_data'])
# compare the instruction and content together, since the same text can be used for different tasks
training_pairs = set(zip(training['user_instruction'], training['clean_data']))
testing_pairs = set(zip(testing['user_instruction'], testing['clean_data']))

display(pd.DataFrame({'check': ['shared source prompt IDs', 'shared task/content pairs', 'shared clean texts'], 'count': [len(shared_ids), len(training_pairs & testing_pairs), len(shared_content)]}).set_index('check'))

# count the records that need further inspection
review_counts = []
for split, data in [('training', training), ('testing', testing)]:
    # check if the original task already contains the answer requested by the injection
    witness_in_task = sum(bool(row.witness) and row.witness.casefold() in row.user_instruction.casefold() for row in data.itertuples())
    review_counts.append({'split': split, 'empty task instructions': data['user_instruction'].str.strip().eq('').sum(), 'records with shared clean text': data['clean_data'].isin(shared_content).sum(), 'witness in task instruction': witness_in_task})

display(pd.DataFrame(review_counts).set_index('split'))

# show five examples where the original user instruction is missing
missing_tasks = training[training['user_instruction'].str.strip().eq('')]
display(missing_tasks[['data_id', 'user_prompt_id', 'user_instruction', 'clean_data']].head(5))
